# 01 — Coleta e Diagnóstico
**Projeto:** churn-upsell-analytics · **Etapa 1 de 3**

Objetivo: carregar o dataset de clientes bancários e entender a estrutura
dos dados antes de qualquer análise de churn ou upsell.

**Dataset:** Churn Modelling (Kaggle) — 10.000 clientes de um banco fictício
com dados demográficos, financeiros e a flag de cancelamento (Exited).


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style("whitegrid")
sns.set_palette("muted")
print("Imports OK")

## 2. Caminhos e carregamento

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_PATH     = PROJECT_ROOT / "data" / "raw" / "Churn_Modelling.csv"

print(f"Arquivo buscado: {RAW_PATH}")
print(f"Existe? {RAW_PATH.exists()}")

df = pd.read_csv(RAW_PATH)
print(f"\nShape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

## 3. O que cada coluna significa

| Coluna | Significado |
|---|---|
| CustomerId | Identificador único do cliente |
| CreditScore | Score de crédito (350-850) |
| Geography | País do cliente (France, Germany, Spain) |
| Gender | Gênero |
| Age | Idade |
| Tenure | Anos como cliente do banco |
| Balance | Saldo em conta |
| NumOfProducts | Quantidade de produtos bancários contratados |
| HasCrCard | Possui cartão de crédito (1) ou não (0) |
| IsActiveMember | Cliente ativo (1) ou inativo (0) |
| EstimatedSalary | Salário estimado |
| **Exited** | **Variável alvo: cliente cancelou (1) ou permaneceu (0)** |


## 4. Tipos de dados e valores nulos

In [ ]:
print("Tipos de dados:")
print(df.dtypes.to_string())
print()

nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if nulls.empty:
    print("Nenhum valor nulo — dataset limpo, como é comum em datasets Kaggle bem curados.")
else:
    print(f"Colunas com nulos:\n{nulls.to_string()}")

## 5. Taxa de churn geral (a métrica mais importante do projeto)

In [ ]:
# Exited é 0 (permaneceu) ou 1 (cancelou)
# .mean() de uma coluna 0/1 = proporção de 1s = taxa de churn

churn_rate = df["Exited"].mean() * 100
n_churned  = df["Exited"].sum()
n_total    = len(df)

print(f"Total de clientes  : {n_total:,}")
print(f"Clientes que saíram: {n_churned:,}")
print(f"Taxa de churn geral: {churn_rate:.1f}%")

fig, ax = plt.subplots(figsize=(5, 4))
df["Exited"].value_counts().rename({0: "Permaneceu", 1: "Cancelou"}).plot(
    kind="bar", ax=ax, color=["#55A868", "#C44E52"], edgecolor="white"
)
ax.set_title("Distribuição de Churn", fontsize=12, pad=10)
ax.set_ylabel("Qtd. clientes")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
(PROJECT_ROOT / "docs" / "screenshots").mkdir(parents=True, exist_ok=True)
plt.savefig(PROJECT_ROOT / "docs" / "screenshots" / "01_churn_distribution.png", dpi=150)
plt.show()

## 6. Estatísticas descritivas

In [ ]:
df.describe().round(2)

## 7. Churn por país — primeiro olhar

In [ ]:
churn_by_geo = df.groupby("Geography")["Exited"].agg(["mean", "count"])
churn_by_geo.columns = ["churn_rate", "total_customers"]
churn_by_geo["churn_rate"] = (churn_by_geo["churn_rate"] * 100).round(1)
churn_by_geo = churn_by_geo.sort_values("churn_rate", ascending=False)

print(churn_by_geo.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
churn_by_geo["churn_rate"].plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("Taxa de Churn por País", fontsize=12, pad=10)
ax.set_ylabel("Taxa de churn (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "screenshots" / "01_churn_by_geography.png", dpi=150)
plt.show()

## 8. Avançar para o notebook 02

In [ ]:
print("=" * 50)
print("  Notebook 01 concluído.")
print("  Próximo: abra 02_tratamento.ipynb")
print("=" * 50)